# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [16]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****2121
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [17]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [18]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [19]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [20]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [21]:
from langchain.agents import create_agent
from azure_openai_setup import get_model_string

agent = create_agent(
    model=get_model_string(),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [23]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_2MZgZAr7AdfBhog9zNhK9C4V)
 Call ID: call_2MZgZAr7AdfBhog9zNhK9C4V
  Args:
    query: SELECT id, total FROM invoices WHERE customer_name = 'Frank Harris' ORDER BY date DESC LIMIT 1
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT id, total FROM invoices WHERE customer_name = 'Frank Harris' ORDER BY date DESC LIMIT 1]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

It seems there is no table named `invoices` in the database. Could you clarify or provide more details about the table structure or the database? For example, do

In [24]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_B56ilucMPvggma1fyEVgBdMc)
 Call ID: call_B56ilucMPvggma1fyEVgBdMc
  Args:
    query: SELECT title FROM table_name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: table_name
[SQL: SELECT title FROM table_name LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

It seems the table name is incorrect or not specified. Could you provide the correct table name or more context about the database?


## Add memory

In [25]:
from langgraph.checkpoint.memory import InMemorySaver

In [26]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=get_model_string(),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [29]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_FlEZBqlz3hLCQD0mvw0Tmkoq)
 Call ID: call_FlEZBqlz3hLCQD0mvw0Tmkoq
  Args:
    query: SELECT * FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(16, 'Frank', 'Harris', 'Google Inc.', '1600 Amphitheatre Parkway', 'Mountain View', 'CA', 'USA', '94043-1351', '+1 (650) 253-0000', '+1 (650) 253-0000', 'fharris@google.com', 4)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_jaWvelzd6uHL68Xv3oDrhB7J)
 Call ID: call_jaWvelzd6uHL68Xv3oDrhB7J
  Args:
    query: SELECT InvoiceId, Total FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
====

In [30]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_rHt0DzCex5nVLw4xp7QGvP1n)
 Call ID: call_rHt0DzCex5nVLw4xp7QGvP1n
  Args:
    query: SELECT Track.Name FROM InvoiceLine INNER JOIN Track ON InvoiceLine.TrackId = Track.TrackId WHERE InvoiceLine.InvoiceId = 374;
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]
================================== Ai Message ==================================

The titles on your last invoice (Invoice ID: 374) were:

1. Holier Than Thou  
2. Through The Never  
3. My Friend Of Misery  
4. The Wait  
5. Blitzkrieg  
6. So What


## Try your own queries
Now that there is memory, check the agents recall!

In [13]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?
================================== Ai Message ==================================

Could you clarify which table might contain the invoice or billing information for Frank Harris? For example, is it stored in a table related to customers, transactions, or another table?
